# Prompt Repetition Benchmark — Analysis Notebook
**Replication of arXiv:2512.14982 for RTL/low-resource Pakistani languages**

Loads results produced by `run_punjabi.py` and provides:
1. Dataset overview & completion check
2. Accuracy table (method × task × scenario)
3. McNemar significance tests
4. Figure 1 — grouped bar chart
5. Sample response explorer
6. Latency & token analysis
7. Export tables to CSV

**To use:** run all cells top-to-bottom. Change `CSV_PATH` in cell 2 to load a specific results file.

In [ ]:
import sys, warnings
from pathlib import Path
from glob import glob

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("module://matplotlib_inline.backend_inline")
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from IPython.display import display

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 90)
pd.set_option("display.float_format", "{:.3f}".format)

# Import analysis functions from analysis.py in the same directory
sys.path.insert(0, str(Path(".").resolve()))
from analysis import (
    load_csv, load_json,
    compute_accuracy, compute_mcnemar,
    print_accuracy_table, print_mcnemar_table, print_summary,
    plot_figure1,
    METHODS, METHOD_COLORS, _short,
)
print("Imports OK")

---
## 1. Load Results

In [ ]:
# Auto-detect most recent results CSV; override CSV_PATH to pick a specific file
candidates = sorted(
    glob("results_pa_*.csv") + glob("results_ur_*.csv"),
    reverse=True,
)

if not candidates:
    raise FileNotFoundError(
        "No results_*.csv found.\n"
        "Run the experiment first: "
        "python run_punjabi.py --models gpt-4o-mini"
    )

print("Available files:")
for i, p in enumerate(candidates):
    kb = Path(p).stat().st_size / 1024
    print(f"  [{i}] {p}  ({kb:.0f} KB)")

CSV_PATH = candidates[0]   # <- change index to pick a different file
print(f"\nLoading: {CSV_PATH}")

df = load_csv(CSV_PATH)

n_err    = df["response"].isna().sum()
err_rate = n_err / len(df) * 100 if len(df) else 0

print(f"\n  Rows       : {len(df):,}")
print(f"  Models     : {df['model'].unique().tolist()}")
print(f"  Tasks      : {sorted(df['task'].unique()) if 'task' in df.columns else 'n/a'}")
print(f"  Scenarios  : {sorted(df['scenario'].unique().tolist())}")
print(f"  Items      : {df['item_id'].nunique()}")
print(f"  API errors : {n_err} ({err_rate:.1f}%)")

---
## 2. Dataset Overview

In [ ]:
# Items per task (baseline only, to avoid double-counting)
if "task" in df.columns:
    task_counts = (
        df[df["method"] == "baseline"]
        .groupby("task")["item_id"]
        .nunique()
        .rename("items")
        .reset_index()
    )
    display(task_counts.style.hide(axis="index").set_caption("Items per task"))

# Overall accuracy per method (all tasks pooled — headline numbers)
overall = (
    df.groupby("method")["is_correct"]
    .agg(n="count", correct="sum")
    .assign(accuracy_pct=lambda x: (x["correct"] / x["n"] * 100).round(1))
    .reindex(METHODS)
)
display(
    overall[["n", "correct", "accuracy_pct"]]
    .style
    .background_gradient(subset="accuracy_pct", cmap="RdYlGn", vmin=0, vmax=100)
    .set_caption("Overall accuracy by method (all tasks pooled)")
)

---
## 3. Accuracy Table
Grouped by `(model, task, scenario, method)`. Δ = method − baseline.

In [ ]:
acc = compute_accuracy(df)
print_accuracy_table(acc)

In [ ]:
# Styled pivot: green = beats baseline, red = worse
pivot = acc.pivot_table(
    index=["model", "task", "scenario"] if "task" in acc.columns
          else ["model", "scenario"],
    columns="method",
    values="accuracy",
    aggfunc="first",
).reindex(columns=METHODS).round(3)

def _highlight_wins(df_raw):
    styles = pd.DataFrame("", index=df_raw.index, columns=df_raw.columns)
    base   = df_raw["baseline"]
    for m in METHODS[1:]:
        if m not in df_raw.columns:
            continue
        styles[m] = [
            "background-color: #d4edda" if v > b
            else ("background-color: #f8d7da" if pd.notna(v) and pd.notna(b) and v < b else "")
            for v, b in zip(df_raw[m], base)
        ]
    return styles

display(
    pivot.style
    .apply(_highlight_wins, axis=None)
    .format("{:.1%}")
    .set_caption("Accuracy by method  (green = beats baseline, red = worse)")
)

---
## 4. McNemar Significance Tests
Paired test per `(model, task, scenario)`.  
`b` = baseline correct & method wrong &nbsp;·&nbsp; `c` = baseline wrong & method correct  
★ p<0.05 &nbsp; ★★ p<0.01 &nbsp; ★★★ p<0.001

In [ ]:
mc = compute_mcnemar(df)
print_mcnemar_table(mc)

In [ ]:
# Significant results only
sig = mc[mc["significant"]].sort_values("p_value")
print(f"{len(sig)} significant result(s) out of {len(mc)} tests")
if len(sig):
    display(
        sig.reset_index(drop=True)
        .style
        .background_gradient(subset="p_value", cmap="Greens_r", vmin=0, vmax=0.05)
        .format({"p_value": "{:.4f}"})
        .set_caption("Significant McNemar results")
    )

In [ ]:
print_summary(acc, mc)

---
## 5. Figure 1 — Grouped Bar Chart

In [ ]:
# Render inline and save to figure1.png
task_col = "task" in acc.columns

task_scs = (
    list(
        acc[["task", "scenario"]]
        .drop_duplicates()
        .sort_values(["task", "scenario"])
        .itertuples(index=False, name=None)
    )
    if task_col
    else [("" , sc) for sc in acc["scenario"].unique()]
)

models   = acc["model"].unique().tolist()
n_sc     = len(task_scs)
bar_w    = 0.14
offsets  = np.linspace(-(len(METHODS)-1)/2, (len(METHODS)-1)/2, len(METHODS)) * bar_w
x        = np.arange(n_sc)

fig, axes = plt.subplots(
    len(models), 1,
    figsize=(max(12, n_sc * 1.8), 4.5 * len(models)),
    squeeze=False,
)
fig.suptitle(
    "Prompt Repetition — Accuracy by Task \u00d7 Scenario & Method",
    fontsize=13, fontweight="bold", y=1.01,
)

sc_labels = [
    f"{t}\n{_short(s)}" if t else _short(s)
    for t, s in task_scs
]

for row_idx, model in enumerate(models):
    ax  = axes[row_idx, 0]
    mdf = acc[acc["model"] == model]
    for m_idx, method in enumerate(METHODS):
        vals = []
        for task, sc in task_scs:
            mask = (mdf["scenario"] == sc) & (mdf["method"] == method)
            if task_col and task:
                mask &= mdf["task"] == task
            sub = mdf[mask]
            vals.append(sub["accuracy"].values[0] * 100 if len(sub) else 0.0)
        bars = ax.bar(
            x + offsets[m_idx], vals,
            width=bar_w * 0.92, color=METHOD_COLORS[method],
            label=method, zorder=2,
        )
        if method != "baseline":
            bvs = []
            for task, sc in task_scs:
                mask = (mdf["scenario"] == sc) & (mdf["method"] == "baseline")
                if task_col and task:
                    mask &= mdf["task"] == task
                sub = mdf[mask]
                bvs.append(sub["accuracy"].values[0] * 100 if len(sub) else 0.0)
            for bar, v, bv in zip(bars, vals, bvs):
                d = v - bv
                if abs(d) >= 0.5:
                    ax.text(
                        bar.get_x() + bar.get_width() / 2,
                        bar.get_height() + 0.8,
                        f"{d:+.0f}",
                        ha="center", va="bottom", fontsize=5.5,
                        color="#1a7a1a" if d > 0 else "#c0392b",
                        fontweight="bold",
                    )
    ax.set_title(f"Model: {model}", fontsize=10, loc="left", pad=4)
    ax.set_xticks(x)
    ax.set_xticklabels(sc_labels, rotation=30, ha="right", fontsize=8)
    ax.yaxis.set_major_formatter(mticker.PercentFormatter())
    ax.set_ylim(0, 110)
    ax.set_ylabel("Accuracy (%)", fontsize=9)
    ax.grid(axis="y", linestyle="--", alpha=0.4, zorder=0)
    ax.spines[["top", "right"]].set_visible(False)

handles = [
    plt.Rectangle((0, 0), 1, 1, color=METHOD_COLORS[m], label=m)
    for m in METHODS
]
fig.legend(
    handles=handles, loc="lower center", ncol=len(METHODS),
    fontsize=9, bbox_to_anchor=(0.5, -0.02), frameon=True,
)
plt.tight_layout(rect=[0, 0.04, 1, 1])
plt.savefig("figure1.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved \u2192 figure1.png")

---
## 6. Sample Response Explorer
Change the config variables to browse different slices.

In [ ]:
# ── Config ──────────────────────────────────────────────────────────────────
TASK_FILTER    = None        # e.g. "GSM8K" | None = all tasks
METHOD_FILTER  = "baseline"  # which method to inspect
CORRECT_FILTER = None        # True = correct only | False = wrong only | None = both
N_SAMPLES      = 10
# ────────────────────────────────────────────────────────────────────────────

mask = df["method"] == METHOD_FILTER
if TASK_FILTER and "task" in df.columns:
    mask &= df["task"] == TASK_FILTER
if CORRECT_FILTER is not None:
    mask &= df["is_correct"] == int(CORRECT_FILTER)

sample = df[mask].sample(min(N_SAMPLES, int(mask.sum())), random_state=42)

cols = [c for c in ["task", "item_id", "scenario", "correct_answer", "response", "is_correct"]
        if c in sample.columns]

display(
    sample[cols].reset_index(drop=True).style
    .applymap(
        lambda v: "background-color: #d4edda" if v == 1
                  else ("background-color: #f8d7da" if v == 0 else ""),
        subset=["is_correct"],
    )
    .set_caption(f"Sample — method={METHOD_FILTER}")
)

In [ ]:
# Head-to-head: items where baseline and another method disagree
COMPARE_METHOD = "repetition"  # <- method to compare against baseline
TASK_H2H       = None          # <- filter by task, or None

base_df = df[df["method"] == "baseline"][
    [c for c in ["item_id", "task", "correct_answer", "response", "is_correct"] if c in df.columns]
]
comp_df = df[df["method"] == COMPARE_METHOD][["item_id", "response", "is_correct"]]

h2h = base_df.merge(comp_df, on="item_id", suffixes=("_base", f"_{COMPARE_METHOD}"))
if TASK_H2H and "task" in h2h.columns:
    h2h = h2h[h2h["task"] == TASK_H2H]

flips = h2h[
    h2h["is_correct_base"] != h2h[f"is_correct_{COMPARE_METHOD}"]
].sample(min(10, len(h2h[h2h["is_correct_base"] != h2h[f"is_correct_{COMPARE_METHOD}"]])), random_state=42)

print(f"Items where baseline \u2260 {COMPARE_METHOD}: {len(flips)} shown")
display(flips.reset_index(drop=True))

---
## 7. Latency & Token Analysis
Verifies the paper's claim: prompt repetition should not inflate output length or latency.

In [ ]:
has_lat = {"output_tokens", "latency_ms"}.issubset(df.columns)

if not has_lat:
    print("No latency/token columns in this CSV.")
else:
    lat = (
        df.groupby("method")[["output_tokens", "latency_ms", "prompt_chars"]]
        .median()
        .reindex(METHODS)
        .round(1)
    )
    lat["out_tok_\u0394"] = (lat["output_tokens"] - lat.loc["baseline", "output_tokens"]).round(1)
    lat["latency_\u0394"]  = (lat["latency_ms"]    - lat.loc["baseline", "latency_ms"]).round(1)
    display(
        lat.style
        .background_gradient(subset="out_tok_\u0394", cmap="RdYlGn_r", vmin=-5, vmax=5)
        .set_caption("Median output tokens & latency per method (\u0394 vs baseline)")
    )

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for ax, col, label in [
        (axes[0], "output_tokens", "Output tokens"),
        (axes[1], "latency_ms",    "Latency (ms)"),
    ]:
        data = [df[df["method"] == m][col].dropna().values for m in METHODS]
        bp = ax.boxplot(
            data, labels=METHODS,
            patch_artist=True,
            medianprops={"color": "black", "linewidth": 2},
        )
        for patch, method in zip(bp["boxes"], METHODS):
            patch.set_facecolor(METHOD_COLORS[method])
            patch.set_alpha(0.7)
        ax.set_title(f"{label} by method", fontsize=11)
        ax.set_ylabel(label)
        ax.tick_params(axis="x", rotation=20)
        ax.spines[["top", "right"]].set_visible(False)

    plt.suptitle("Output length & latency (no CoT inflation expected)", fontsize=12, fontweight="bold")
    plt.tight_layout()
    plt.savefig("latency_analysis.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved \u2192 latency_analysis.png")

---
## 8. Export

In [ ]:
acc.to_csv("accuracy_table.csv", index=False)
mc.to_csv("mcnemar_results.csv",  index=False)
print("Saved: accuracy_table.csv")
print("Saved: mcnemar_results.csv")
print("Saved: figure1.png  (generated in section 5)")